# Reproducing Kay et al. (2008) from CRCNS vim-1

**Paper.** Kay KN, Naselaris T, Prenger RJ, Gallant JL (2008). Identifying natural images from human
brain activity. *Nature* 452:352-355. [doi:10.1038/nature06713](https://doi.org/10.1038/nature06713).
The claims file `claims/kay-2008.yaml` was written from the version of record with its
Supplementary Information (Supplementary Figures 1 to 11, Supplementary Table 1, Supplementary
Discussion 1 to 3, Supplementary Methods 1 to 11; SM below).

**Data set.** CRCNS [vim-1](http://crcns.org/data-sets/vc/vim-1), doi:10.6080/K0QN64NG, read
anonymously from the public bucket `crcnsarchive` (`vim-1/`, no `live/` folder). CRCNS data sets are
not versioned, so the version is recorded as `unversioned` (docs/archives.md); the paper cites no
version, and the bucket holds a single copy, the one documented by the data set's readme version
1.2 of 2013-04-09. The MD5 sums of the files read, from the data set's `checksums.md5`, are:

| File | MD5 |
|---|---|
| EstimatedResponses.mat | 7b702b34d42ba612118d6ad99e86e217 |
| Stimuli.mat | 464676501cb768b005f93a32c70e73c2 |
| Stimuli_Trn_FullRes_01.mat to _15.mat | listed in the data check below |
| Stimuli_Val_FullRes.mat | 2e7d70ecff050d8c74b066b35e5bc00c |
| mask.mat | 19e624419b35eaed38534047665c526c |
| Sub1_Ses1_Run1_Val.nii.gz (timing estimate only) | listed in the data check below |

**What the data set holds.** For each of the two subjects, the repeated-trial responses
(estimated per image, z-scored per voxel) of about 26,000 voxels to the 1,750 model estimation
images and the 120 identification images, ROI labels, the stimuli at 128 px and at full
resolution, the presentation sequences, the circular stimulus mask, anatomy, and the raw time series
of the five sessions (70 runs). It does not hold the later sessions of S1, the 999-image library
of the set-size analyses, the retinotopic mapping scans, single-trial responses, or the standard
errors of the response estimates.

**Method.** The identification pipeline of the claims file (P1 to P5) is in `vim1.py`, whose
docstring says where each parameter comes from. Two departures apply to every tested claim: the
paper's 100 bootstrap fits per model are replaced by one fit (10 bootstrap fits are shown as a
variant), and one stopping set is drawn per fit and shared by the voxels fitted together.

| Claim | Data | Status |
|---|---|---|
| fig3-s1-repeated-120 | dataset | tested |
| fig4a-s2-repeated-120 | dataset | tested |
| fig4a-single-trial | dataset | blocked: too large for the time left (estimate below) |
| fig4a-noise-ceiling | dataset | blocked: depends on fig4a-single-trial |
| fig4b-set-size-1000 | with inputs | blocked: the 999-image library is not public |
| fig4b-extrapolation | with inputs | blocked: needs the library |
| fig4c-retinotopy-only-extrapolation | with inputs | blocked: needs the library |
| suppm8-retinotopy-only-120 | dataset | tested |
| suppfig8-constrained-models | with inputs | blocked: needs the library |
| time-two-months | dataset | blocked: sessions not in the data set |
| time-fourteen-months | dataset | blocked: session not in the data set |
| errors-similar-images | dataset | tested |
| suppfig4-500-voxels | dataset | tested on repeated trials; the single-trial conditions are blocked |
| supptable1-voxel-counts | dataset | tested |
| supptable1-predictive-power-by-area | dataset | tested |
| supptable1-snr | dataset | blocked: too large for the time left |
| methods-v4-poor | dataset | tested |
| fig2-representative-voxel | dataset | tested |
| suppfig7-roi-tuning | dataset | tested |
| suppfig10-retinotopy-validation | dataset | blocked: mapping data not in the data set |
| suppfig11-rf-size-eccentricity | dataset | tested |
| suppdisc1-classification-extension | dataset | tested |
| suppm7-selected-voxels-v1 | dataset | tested |

In [ ]:
import os
for var in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ.setdefault(var, "4")
import io, gzip, time, resource
import numpy as np
import fsspec
import matplotlib.pyplot as plt
from scipy import stats
import vim1

STARTED = time.time()
def peak_gb():
    return resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6
print("data set", vim1.DATASET, "version", vim1.VERSION)

## Data check

The MD5 sums are read from the data set's `checksums.md5` for every file this notebook reads.

In [ ]:
with fsspec.open(vim1.BASE + "checksums.md5", "rt") as f:
    md5 = dict(reversed(line.split()) for line in f if line.strip())
read = (["EstimatedResponses.mat", "Stimuli.mat", "mask.mat", "Stimuli_Val_FullRes.mat",
         "Sub1_Ses1_Run1_Val.nii.gz"] + [f"Stimuli_Trn_FullRes_{k:02d}.mat" for k in range(1, 16)])
for name in read:
    print(f"{name:30s} {md5[name]}")

In [ ]:
t = time.time()
S = vim1.load_stimuli()
full = S["full"]
raw_min, raw_max = float(full.min()), float(full.max())
MU = float(full.mean())
full -= MU                       # mean luminance over all stimuli subtracted (SM 6)
FILL = -MU                       # the grey background (0 in the files) after that subtraction
RADIUS = vim1.stimulus_radius_deg(S["mask"])
print(f"stimuli loaded in {time.time() - t:.0f} s; full resolution {full.shape}, values {raw_min:.3f} to {raw_max:.3f}, "
      f"mean {MU:.4f}; 128 px {S['small'].shape}")
print("corner pixel of every full-resolution image is 0 (grey background):", bool((full[:, 0, 0] == FILL).all()))
r128 = vim1.resize_stack(full[:200] + MU, 128)
print("full resolution resampled to 128 px against Stimuli.mat, r =",
      round(float(np.corrcoef(r128.ravel(), S["small"][:200].ravel())[0, 1]), 4),
      "(transposed:", round(float(np.corrcoef(r128.transpose(0, 2, 1).ravel(), S["small"][:200].ravel())[0, 1]), 3), ")")
r128v = vim1.resize_stack(full[1750:] + MU, 128)
print("validation images, same comparison, r =", round(float(np.corrcoef(r128v.ravel(), S["small"][1750:].ravel())[0, 1]), 4))
del r128, r128v
m = S["mask"]
print(f"mask {m.shape}, values {m.min()} to {m.max()}; radius from its area {RADIUS:.2f} deg of 20 "
      f"(stimulus diameter {2 * RADIUS:.1f} deg)")
seq_trn, seq_val = S["seq_trn"], S["seq_val"]
print("training runs:", seq_trn.shape[1], "of", seq_trn.shape[0], "s; image onsets per run",
      sorted(set((seq_trn > 0).sum(0))), "; distinct images per run", sorted({len(np.unique(c[c > 0])) for c in seq_trn.T}),
      "; presentations per image", sorted(set(np.bincount(seq_trn[seq_trn > 0])[1:])),
      "; onsets 4 s apart:", bool(all((np.flatnonzero(c)[:, None] % 4 == np.flatnonzero(c)[0] % 4).all() for c in seq_trn.T)))
print("identification runs:", seq_val.shape[1], "of", seq_val.shape[0], "s; image onsets per run",
      sorted(set((seq_val > 0).sum(0))), "; distinct images per run", sorted({len(np.unique(c[c > 0])) for c in seq_val.T}),
      "; presentations per image over all runs", sorted(set(np.bincount(seq_val[seq_val > 0])[1:])))
first6 = [(seq_val[:24:4, j] == seq_val[-24::4, j]).all() for j in range(10)]
print("first 6 trials repeated at the end of each identification run:", first6)
print("images per session (identification): ", [len(np.unique(seq_val[:, 2 * s:2 * s + 2][seq_val[:, 2 * s:2 * s + 2] > 0])) for s in range(5)])
print("training images over all runs:", len(np.unique(seq_trn[seq_trn > 0])))
for name in ("Stimuli.mat", "mask.mat", "Stimuli_Val_FullRes.mat"):
    with vim1._open(name) as f:
        print(f"{name} header: {f.read(116).decode('ascii', 'replace').strip()[:40]!r}")

In [ ]:
DATA = {}
for subj in ("S1", "S2"):
    D = vim1.load_responses(subj)
    DATA[subj] = D
    print(subj, "voxels in the file:", D["n_total"], "; per ROI:", D["counts_all"])
    print("   kept V1, V2, V3, V4 (no NaN):", {vim1.ROI_NAMES[k]: int((D['roi'] == k).sum()) for k in (1, 2, 3, 6)},
          "; dropped for NaN:", D["dropped_nan"])
    tr, va = D["trn"], D["val"]
    print(f"   training responses {tr.shape}: per-voxel mean {np.abs(tr.mean(0)).max():.2e} (max abs), "
          f"SD {tr.std(0).min():.3f} to {tr.std(0).max():.3f}, range {tr.min():.1f} to {tr.max():.1f}")
    print(f"   identification responses {va.shape}: per-voxel mean {va.mean(0).min():.2f} to {va.mean(0).max():.2f}, "
          f"SD {va.std(0).min():.2f} to {va.std(0).max():.2f}")
    print(f"   voxel index range {D['vox_idx'].min()} to {D['vox_idx'].max()} (64 x 64 x 18 = {64 * 64 * 18})")
d1 = DATA["S1"]
w = np.flatnonzero(d1["vox_idx"] == 42205)
print("S1 voxel index 42205:", "present, ROI", [vim1.ROI_NAMES[d1['roi'][i]] for i in w] if w.size else "absent")
print(f"peak memory {peak_gb():.1f} GB")

The training responses are z-scored per voxel, as the readme says (mean 0, SD 1 to rounding). The
identification responses are not z-scored on their own; their means and SDs vary by voxel, which
fits the paper's statement that the training responses' mean and SD were applied to them (SM 3).
The readme says the `.mat` files are MATLAB 7.3; `Stimuli.mat` and `mask.mat` are MATLAB 5 files
(the full-resolution files are 7.3), and it gives `Stimuli_Trn_FullRes_15.mat` as 70 x 120 x 120,
which is 70 x 500 x 500. The stimulus sequences and voxel counts are checked against the paper in
the output above. Whether the responses and stimuli share their image order is checked below by
predicting the identification responses with the images in their order and in shuffled orders.

## The identification pipeline

Per subject: the three orientation-invariant initial models for every V1, V2, V3 and V4 voxel; the
best initial model and its predictive power and bootstrap p-value; final models under O5 (b) for
every V1 to V3 voxel with a best initial model (O1 b), and under O5 (a) for the same voxels, one
model per voxel. Each voxel's final model is fitted on its own, so the O1 (a) readings use the
same models restricted to the voxels whose best initial model has p < 0.01.

In [ ]:
t = time.time()
chans_init, M_init = vim1.initial_channels(full, FILL, RADIUS)
chans_b, keeps_b, W64 = vim1.final_b_channels(full, FILL, RADIUS)
print(f"channels in {time.time() - t:.0f} s:", {k: v.shape for k, v in chans_init.items()}, "initial;",
      {k: v.shape for k, v in chans_b.items()}, "final (O5 b)")
print("fields of view of the initial models (deg):", {k: round(v[1], 2) for k, v in vim1.FOVS.items()})


def run_subject(subj, seed):
    rng = np.random.default_rng(seed)
    D = DATA[subj]
    out = dict(D=D)
    t = time.time()
    init = vim1.initial_models(D, chans_init, M_init, rng)
    out["init"] = init
    out["t_init"] = time.time() - t
    v13 = np.isin(D["roi"], [1, 2, 3])
    out["candA"] = v13 & (init["p"] < 0.01)
    out["candB"] = v13 & init["has"]
    t = time.time()
    out["pred_b"], out["kern_b"] = vim1.final_b(D, init, chans_b, rng, out["candB"])
    out["t_b"] = time.time() - t
    t = time.time()
    out["pred_a"], out["kern_a"] = vim1.final_a(D, init, full, FILL, RADIUS, rng, out["candB"], W64)
    out["t_a"] = time.time() - t
    return out


RES = {}
for subj, seed in (("S1", 1), ("S2", 2)):
    RES[subj] = R = run_subject(subj, seed)
    init = R["init"]
    print(f"{subj}: initial models {R['t_init']:.0f} s, final (b) {R['t_b']:.0f} s for {R['candB'].sum()} voxels, "
          f"final (a) {R['t_a']:.0f} s for {R['candB'].sum()} voxels ({R['t_a'] / max(R['candB'].sum(), 1):.2f} s each); "
          f"peak memory {peak_gb():.1f} GB")
    for name in vim1.FOVS:
        mm = init["models"][name]
        print(f"   initial {name:>2s} deg: fits that stopped at iteration 0 (no kernel) {int((mm['iters'] == 0).sum())} of "
              f"{mm['iters'].size}; iterations median {np.median(mm['iters']):.0f}, max {mm['iters'].max()}; valid RF {int(mm['valid'].sum())}")
    print("   best initial model:", {n: int((init['best'][init['has']] == n).sum()) for n in vim1.FOVS},
          "; voxels with none:", int((~init['has']).sum()))

### Do the responses and the stimuli share their order?

The best initial model's predictions are correlated with the identification responses in the
images' order, and in 20 random permutations of the image order.

In [ ]:
rng_check = np.random.default_rng(7)
for subj in ("S1", "S2"):
    R = RES[subj]; D = R["D"]; v1 = (D["roi"] == 1) & R["init"]["has"]
    true_r = np.median(vim1.pearson_cols(D["val"][:, v1], R["init"]["pred"][:, v1]))
    perm = [np.median(vim1.pearson_cols(D["val"][rng_check.permutation(120)][:, v1], R["init"]["pred"][:, v1]))
            for _ in range(20)]
    print(f"{subj}: median r over {v1.sum()} V1 voxels, image order {true_r:.3f}; permuted orders {min(perm):.3f} to {max(perm):.3f}")

Responses and stimuli line up in the order the readme gives: the shuffled orders give median
correlations near zero.

## fig3-s1-repeated-120 and fig4a-s2-repeated-120

> For subject S1, the image seen was identified correctly for 92% (110 of 120) of the
> repeated-trial voxel activity patterns from 120 novel natural images, against chance of 0.8%.
> For subject S2, 72% (86 of 120) of the images were identified correctly from repeated-trial
> patterns.

Criterion: P1 to P5; the values must hold under one of the readings of O1 (candidate voxels:
(a) V1 to V3 voxels whose best initial model has p < 0.01, (b) all V1 to V3 voxels) and O5 (final
model field of view: (a) the +/-2 SD square, (b) the best initial model's field of view). Expected
92 +/- 9.2 (S1) and 72 +/- 7.2 (S2) percent correct.

In [ ]:
READINGS = {"O1a-O5a": ("pred_a", "candA"), "O1a-O5b": ("pred_b", "candA"), "O1b-O5a": ("pred_a", "candB"),
            "O1b-O5b": ("pred_b", "candB")}
ID = {}
for subj in ("S1", "S2"):
    R = RES[subj]; D = R["D"]
    for name, (pk, ck) in READINGS.items():
        pred = R[pk]
        power = vim1.loo_power(D["val"], pred)
        choice, sel = vim1.identify_repeated(D["val"], pred, power, cand=R[ck])
        ID[subj, name] = dict(choice=choice, sel=sel, power=power, pred=pred, cand=R[ck])
        n_ok = int((choice == np.arange(120)).sum())
        print(f"{subj} {name}: {n_ok}/120 = {100 * n_ok / 120:.1f}% correct (candidate voxels {R[ck].sum()}; "
              f"binomial p against chance 1/120 = {stats.binomtest(n_ok, 120, 1 / 120, alternative='greater').pvalue:.1e})")
    # the best initial model alone, for reference
    power = vim1.loo_power(D["val"], R["init"]["pred"])
    choice, _ = vim1.identify_repeated(D["val"], R["init"]["pred"], power, cand=R["candA"])
    print(f"{subj} best initial models only (O1 a): {(choice == np.arange(120)).sum()}/120")

Variant: the paper's bootstrap averaging (SM 6, "Estimation of variance"), with 10 bootstrap fits
in place of 100, for the final models under O5 (b) on the O1 (a) voxels.

In [ ]:
for subj, seed in (("S1", 11), ("S2", 12)):
    R = RES[subj]; D = R["D"]
    t = time.time()
    pred_boot, _ = vim1.final_b(D, R["init"], chans_b, np.random.default_rng(seed), R["candA"], n_boot=10)
    power = vim1.loo_power(D["val"], pred_boot)
    choice, _ = vim1.identify_repeated(D["val"], pred_boot, power, cand=R["candA"])
    n_ok = int((choice == np.arange(120)).sum())
    print(f"{subj} O1a-O5b with 10 bootstrap fits: {n_ok}/120 = {100 * n_ok / 120:.1f}% ({time.time() - t:.0f} s)")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 4))
for ax, subj in zip(axes, ("S1", "S2")):
    I = ID[subj, "O1a-O5a"]; D = RES[subj]["D"]; sel = I["sel"]
    rmat = np.array([[np.corrcoef(D["val"][j, sel[j]], I["pred"][i, sel[j]])[0, 1] for i in range(120)] for j in range(120)])
    ax.imshow(rmat, cmap="viridis"); ax.set_xlabel("predicted pattern (image)"); ax.set_ylabel("measured pattern (image)")
    ax.set_title(f"{subj}, O1a-O5a: r from {rmat.min():.2f} to {rmat.max():.2f}")
plt.tight_layout(); plt.show()

**Result: reproduced (both claims).** Under O1 (a) with O5 (a), 109 of 120 images (90.8%) are
identified for S1 and 87 of 120 (72.5%) for S2, against 110 and 86 in the paper and chance of 1 in
120; both are within tolerance. O1 (b) with O5 (a) gives 110 (91.7%) and 87 (72.5%). Under O5 (b), with the final model over the best initial model's
field of view, performance is 75.8% (S1) and 46.7% (S2) for both readings of O1, outside tolerance,
and below the best initial models alone (96 and 77 of 120). Ten bootstrap fits change O5 (b) by at
most three images (90 and 59 of 120).

## Single-trial claims: what it would take

fig4a-single-trial, fig4a-noise-ceiling (which depends on it) and supptable1-snr need responses the
data set does not hold: single-trial amplitudes, or the standard errors of the repeated-trial
amplitudes. Both come from fitting the BRS model (SM 3 and 4) to the time series of every voxel:
per session, a repeated-trial fit with a Fourier basis to get each voxel's HRF, then a fit with one
amplitude per trial (1,024 trials and 28 polynomial terms on 4,752 time points per session) with
first-order autoregressive noise, iterated to convergence. Each voxel has its own HRF and noise
model, so each needs its own least-squares fits. The cell below streams one run and times one
single-trial least-squares solve of that size.

In [ ]:
t = time.time()
with fsspec.open(vim1.BASE + "Sub1_Ses1_Run1_Val.nii.gz", block_size=2**23).open() as f:
    raw_bytes = f.read()
import nibabel as nib
img = nib.Nifti1Image.from_bytes(gzip.decompress(raw_bytes))
t_stream = time.time() - t
print(f"one run streamed and decompressed in {t_stream:.1f} s: shape {img.shape}, {img.get_data_dtype()}, "
      f"voxel size {img.header.get_zooms()[:3]} mm, TR field {img.header.get_zooms()[3]}")
n_tp = 5 * seq_trn.shape[0] + 2 * seq_val.shape[0]
n_trials = 5 * int((seq_trn[:, 0] > 0).sum()) + 2 * int((seq_val[:, 0] > 0).sum())
n_reg = n_trials + 4 * 7
A = np.random.default_rng(0).standard_normal((n_tp, n_reg)).astype(np.float32)
y = np.random.default_rng(1).standard_normal((n_tp, 1)).astype(np.float32)
t = time.time()
for _ in range(3):
    np.linalg.lstsq(A, y, rcond=None)
t_solve = (time.time() - t) / 3
n_vox = sum(int(np.isin(RES[s]["D"]["roi"], [1, 2, 3]).sum()) for s in ("S1", "S2"))
n_iter = 3
est = n_vox * 5 * n_iter * t_solve / 60 + 70 * t_stream / 60
n_tp_r = 5 * seq_trn.shape[0]; n_reg_r = 350 + 4 * 5
A_r = np.random.default_rng(2).standard_normal((n_tp_r, n_reg_r)).astype(np.float32)
t = time.time()
for _ in range(3):
    np.linalg.lstsq(A_r, y[:n_tp_r], rcond=None)
t_solve_r = (time.time() - t) / 3
est_r = n_vox * 5 * n_iter * 2 * t_solve_r / 60 + 50 * t_stream / 60
print(f"repeated-trial fit (for the SNR), one solve of {n_tp_r} x {n_reg_r}: {t_solve_r:.3f} s; with the alternating HRF step "
      f"(counted as a second solve) over {n_vox} voxels x 5 sessions x {n_iter} iterations + streaming 50 runs: about {est_r:.0f} minutes")
print(f"one solve of {n_tp} x {n_reg}: {t_solve:.2f} s; {n_vox} V1 to V3 voxels x 5 sessions x {n_iter} iterations "
      f"(the least the autoregressive procedure needs) + streaming 70 runs: about {est:.0f} minutes, "
      f"before the repeated-trial HRF fits")
del raw_bytes, img, A

The estimate printed above scales a dense solve of the whole design to every V1 to V3 voxel and to
the model estimation runs as well, so it overstates the cost: the trials of different runs do not
share regressors, and the single-trial identification needs only the identification runs of the
candidate voxels. These claims stay blocked as `too_large` in this revision: a faster fit of the
time series was written but not run in this notebook within the time of the revision.

## supptable1-voxel-counts

> The visual areas hold 1331 (V1), 2208 (V2) and 1973 (V3) voxels in S1, and 1513, 1982 and 1780 in S2.

Criterion: counts of voxels assigned to each area; reproduced when all six are within 10%.

In [ ]:
paper_counts = {"S1": (1331, 2208, 1973), "S2": (1513, 1982, 1780)}
for subj in ("S1", "S2"):
    c = RES[subj]["D"]["counts_all"]
    for area, exp in zip(("V1", "V2", "V3"), paper_counts[subj]):
        print(f"{subj} {area}: {c[area]} voxels labelled in the data set (paper {exp}; difference {c[area] - exp})")

**Result: reproduced.** All six counts printed above equal the paper's.

## supptable1-predictive-power-by-area

> The fraction of voxels whose best initial model has significant predictive power falls from V1
> to V2 to V3 (S1 40%, 31%, 17%; S2 25%, 21%, 13%).

Criterion: best initial model, p < 0.01 by the bootstrap (O2), as a percentage of each area's
voxels; reproduced when V1 > V2 > V3 in both subjects and at least four of six values are within
10%.

In [ ]:
paper_frac = {"S1": (40, 31, 17), "S2": (25, 21, 13)}
FRAC = {}
for subj in ("S1", "S2"):
    R = RES[subj]; D = R["D"]
    for area, k, exp in zip(("V1", "V2", "V3"), (1, 2, 3), paper_frac[subj]):
        n_sig = int(((D["roi"] == k) & (R["init"]["p"] < 0.01)).sum())
        tot = D["counts_all"][area]
        FRAC[subj, area] = 100 * n_sig / tot
        within = abs(FRAC[subj, area] - exp) <= 0.1 * exp
        print(f"{subj} {area}: {n_sig} of {tot} = {FRAC[subj, area]:.1f}% (paper {exp}%, within 10%: {within}); "
              f"of the {(D['roi'] == k).sum()} voxels without NaN {100 * n_sig / (D['roi'] == k).sum():.1f}%; "
              f"median r {np.median(R['init']['r'][D['roi'] == k]):.3f}")
    print(f"{subj} order V1 > V2 > V3: {FRAC[subj, 'V1'] > FRAC[subj, 'V2'] > FRAC[subj, 'V3']}")
    for thr in (0.001, 0.05):
        print(f"   variant p < {thr}:", [round(100 * float(((D['roi'] == k) & (R['init']['p'] < thr)).sum()) / D['counts_all'][a], 1)
                                    for a, k in (("V1", 1), ("V2", 2), ("V3", 3))])
print("values within 10%:", sum(abs(FRAC[s, a] - e) <= 0.1 * e for s in ("S1", "S2") for a, e in zip(("V1", "V2", "V3"), paper_frac[s])), "of 6")

**Result: reproduced.** The fractions are 41.2%, 30.2% and 18.2% (S1) and 26.8%, 22.6% and 13.0%
(S2), all six within 10% of the paper's values, and V1 > V2 > V3 in both subjects. The denominators
are all voxels the data set labels in each area; 345 (S1) and 214 (S2) of them have NaN responses
and count as not significant. This is an assumption: Supplementary Table 1 gives the fractions as
percentages of these totals, which equal the label counts including the NaN voxels. Over the voxels
without NaN only, 3 of the 6 values are within 10% (S1 V3 20.1%, S2 V1 29.0%, S2 V2 23.7% are not),
which would be partially reproduced.

## methods-v4-poor

> The GWP model characterizes the responses of voxels in V1, V2 and V3 but does poorly for higher
> areas such as V4.

Criterion: the percentage of voxels whose best initial model has p < 0.01, for V4 (the data set's
labels) and V1 to V3; reproduced when the V4 percentage is below the V3 percentage in both subjects.

In [ ]:
for subj in ("S1", "S2"):
    R = RES[subj]; D = R["D"]
    row = []
    for area, k in (("V1", 1), ("V2", 2), ("V3", 3), ("V4", 6)):
        s = D["roi"] == k
        row.append(f"{area} {100 * ((R['init']['p'] < 0.01) & s).sum() / D['counts_all'][area]:.1f}% "
                   f"(median r {np.median(R['init']['r'][s]):.3f})")
    v3 = ((R['init']['p'] < 0.01) & (D['roi'] == 3)).sum() / D['counts_all']['V3']
    v4 = ((R['init']['p'] < 0.01) & (D['roi'] == 6)).sum() / D['counts_all']['V4']
    print(subj, "; ".join(row), f"; V4 below V3: {v4 < v3}")
# variant: the initial models refitted with other seeds (stopping set and bootstrap p-values)
for subj, seeds in (("S1", (101, 102)),):          # S1, where the margin is small
    D = RES[subj]["D"]
    for seed in seeds:
        i2 = vim1.initial_models(D, chans_init, M_init, np.random.default_rng(seed))
        fr = {a: 100 * ((D["roi"] == k) & (i2["p"] < 0.01)).sum() / D["counts_all"][a] for a, k in (("V1", 1), ("V2", 2), ("V3", 3), ("V4", 6))}
        print(f"{subj} seed {seed}: " + "; ".join(f"{a} {x:.1f}%" for a, x in fr.items()) + f"; V4 below V3: {fr['V4'] < fr['V3']}")
        del i2

**Result: reproduced.** V4 voxels with significant predictive power are 17.5% (S1) and 8.6% (S2),
below V3 (18.2% and 13.0%) in both subjects; the S1 margin is small (0.7 points). With the initial
models of S1 refitted under two other seeds, V4 stays below V3 (16.6% against 18.6%, and 16.8%
against 17.4%). Median r is 0.019 (S1) and 0.000 (S2) in V4, against 0.135 and 0.069 in V1.

## suppm7-selected-voxels-v1

> Most of the 500 voxels used for identification were located in V1, where predictive power was highest.

Readings: (a) voxels selected in more than half of the 120 selections, (b) the top 500 by
predictive power over all 120 images; reproduced when more than half are in V1 in both subjects.
Shown for every identification reading; O1a-O5a is the first.

In [ ]:
for name in READINGS:
    for subj in ("S1", "S2"):
        I = ID[subj, name]; D = RES[subj]["D"]
        counts = np.bincount(np.concatenate(I["sel"]), minlength=D["roi"].size)
        a = np.flatnonzero(counts > 60)
        r_all = np.where(I["cand"], np.nan_to_num(vim1.pearson_cols(D["val"], I["pred"]), nan=-1), -np.inf)
        b = np.argsort(-r_all)[:500]
        fa = [(D["roi"][a] == k).mean() for k in (1, 2, 3)]
        fb = [(D["roi"][b] == k).mean() for k in (1, 2, 3)]
        print(f"{name} {subj}: (a) {a.size} voxels, V1/V2/V3 = {fa[0]:.2f}/{fa[1]:.2f}/{fa[2]:.2f}; "
              f"(b) V1/V2/V3 = {fb[0]:.2f}/{fb[1]:.2f}/{fb[2]:.2f}")

**Result: not reproduced.** Under the first identification reading (O1a-O5a), 48% of the selected
voxels lie in V1 in both subjects, under both readings of the selection (V2 41 to 42%, V3 10 to
11%); under O5 (b) it is 42 to 43%. V1 holds the largest share but not more than half.

## errors-similar-images

> Identification errors tended to occur when the selected image was visually similar to the correct image.

Criterion: for each error of the S1 and S2 repeated-trial identification, the percentile of the
selected image's similarity to the correct image among the correct image's similarities to the
other 119; readings (a) pixel luminance correlation, (b) correlation of local RMS contrast in
0.6 degree blocks; reproduced when, under one reading, the pooled errors have a median percentile
above 50 by a one-sided Wilcoxon signed-rank test at p < 0.05. Main identification reading O1a-O5a,
others shown.

In [ ]:
val_raw = full[1750:] + MU
sim_lum = vim1.corr_rows(val_raw.reshape(120, -1), val_raw.reshape(120, -1))
BLOCK = round(0.6 * vim1.FULL_PX / vim1.STIM_DEG)
con_val = vim1.local_contrast(val_raw, BLOCK)
sim_con = vim1.corr_rows(con_val, con_val)
print(f"blocks of {BLOCK} px ({BLOCK * vim1.STIM_DEG / vim1.FULL_PX:.2f} deg), {con_val.shape[1]} per image")


def error_percentiles(choice, sim):
    out = []
    for j in np.flatnonzero(choice != np.arange(120)):
        others = np.delete(sim[j], j)
        out.append(100 * (others < sim[j, choice[j]]).mean())
    return np.array(out)


for name in READINGS:
    for sname, sim in (("(a) luminance", sim_lum), ("(b) local contrast", sim_con)):
        pct = np.concatenate([error_percentiles(ID[s, name]["choice"], sim) for s in ("S1", "S2")])
        w = stats.wilcoxon(pct - 50, alternative="greater")
        per = {s: np.median(error_percentiles(ID[s, name]["choice"], sim)) for s in ("S1", "S2")}
        print(f"{name} {sname}: {pct.size} errors, median percentile {np.median(pct):.1f} "
              f"(S1 {per['S1']:.1f}, S2 {per['S2']:.1f}); Wilcoxon one-sided p = {w.pvalue:.2g}")
rng_null = np.random.default_rng(21)
null = [np.median(np.concatenate([error_percentiles(rng_null.integers(0, 120, 120), sim_lum)])) for _ in range(200)]
print(f"null (random choices), median percentile over 200 draws: {np.median(null):.1f} (5th to 95th {np.percentile(null, 5):.1f} to {np.percentile(null, 95):.1f})")

**Result: reproduced, under reading (b).** With local contrast as the similarity, the 44 errors of
O1a-O5a have a median percentile of 68.9 (S1 93.3, S2 64.7), one-sided Wilcoxon p = 0.00031; with
pixel luminance the median is 42.4 (p = 0.7). The other identification readings give the same
pattern (contrast 68.9, 61.3 and 63.9, p <= 0.00015; luminance p >= 0.7). Random choices give a median
near 50 (49.6).

## suppdisc1-classification-extension

> Extending classification to novel images works poorly, 1.7% and 0.8% correct with pixel
> luminance similarity (not above chance) and 5% and 5.8% with local contrast (above chance), for
> S1 and S2.

Criterion: match each identification pattern to the most correlated of the 1,750 training
patterns, then pick the identification image most similar to that training image. Voxels (a) the
500 voxels of P5 for that pattern (O1a-O5a), (b) all candidate V1 to V3 voxels (O1 a and b). RMS
contrast (a) SD in the block, (b) SD over the block mean. Reproduced when, under one reading, each
count is within one image of 2, 1, 6, 7, the luminance results have binomial p >= 0.05 and the
contrast results p < 0.0001.

In [ ]:
trn_raw = full[:1750] + MU
sim_trn_lum = vim1.corr_rows(trn_raw.reshape(1750, -1), val_raw.reshape(120, -1))     # 1750 x 120
con_trn = vim1.local_contrast(trn_raw, BLOCK)
sim_trn_con = vim1.corr_rows(con_trn, con_val)
shift = -raw_min   # values shifted to start at 0 for the relative contrast (absolute luminance not given)
con_trn_rel = vim1.local_contrast(trn_raw + shift, BLOCK, relative=True)
con_val_rel = vim1.local_contrast(val_raw + shift, BLOCK, relative=True)
sim_trn_rel = vim1.corr_rows(con_trn_rel, con_val_rel)
del trn_raw
paper = {("S1", "lum"): 2, ("S2", "lum"): 1, ("S1", "con"): 6, ("S2", "con"): 7}
for vname in ("(a) 500 voxels of P5", "(b) O1a voxels", "(b) O1b voxels"):
    for subj in ("S1", "S2"):
        D = RES[subj]["D"]; R = RES[subj]
        matched = np.empty(120, int)
        if not vname.startswith("(a)"):
            vs = np.flatnonzero(R["candA"] if "O1a" in vname else R["candB"])
            matched = np.argmax(vim1.corr_rows(D["trn"][:, vs].copy(), D["val"][:, vs].copy()), 0)
        for j in range(120 if vname.startswith("(a)") else 0):
            vs = ID[subj, "O1a-O5a"]["sel"][j]
            m = D["val"][j, vs] - D["val"][j, vs].mean()
            t_ = D["trn"][:, vs] - D["trn"][:, vs].mean(1, keepdims=True)
            matched[j] = np.argmax(t_ @ m / np.linalg.norm(t_, axis=1))
        line = []
        for mname, sim, key in (("luminance", sim_trn_lum, "lum"), ("contrast SD", sim_trn_con, "con"),
                                ("contrast SD/mean", sim_trn_rel, "con")):
            pick = np.argmax(sim[matched], 1)
            k_ = int((pick == np.arange(120)).sum())
            p_ = stats.binomtest(k_, 120, 1 / 120, alternative="greater").pvalue
            line.append(f"{mname} {k_}/120 (paper {paper[subj, key]}, p = {p_:.2g})")
        print(f"{vname} {subj}: " + "; ".join(line))

**Result: not reproduced.** The luminance metric is not above chance under any reading (0 to 3 of
120, p >= 0.08), as in the paper. The contrast metric is above chance at p < 0.0001 only for S1 with
the 500 voxels of P5 and the SD measure (9 of 120, p = 8.8e-07); for S2 it is 1 or 2 of 120
(p >= 0.26) under every reading, against 7 in the paper. No reading gives the paper's pattern.

## suppm8-retinotopy-only-120

> At set size 120 with repeated trials, averaged across subjects, the retinotopy-only model
> identified 55% of images with spatially-weighted luminance and contrast and 42% with the standard
> (unweighted) metrics, both below the Gabor wavelet pyramid model.

Criterion: the RO model (SM 8) at the best initial model's Gaussian, fitted by the gradient
descent of SM 6, voxels by its own predictive power (SM 7), identification by P5, the two
subjects averaged; reproduced when both values are within 10% (55 +/- 5.5, 42 +/- 4.2) and below the
GWP mean computed here. The candidate voxels follow O1 (a) and (b).

In [ ]:
RO = {}
# luminance relative to the mean luminance of all stimuli, as SM 8 defines it ("SM 8"), and relative
# to the grey background, the reading of the first version of this notebook ("grey")
for lum, base, seeds in (("SM 8", full, (33, 34)), ("grey", full + MU, (31, 32))):
    for subj, seed in zip(("S1", "S2"), seeds):
        R = RES[subj]; D = R["D"]; rng = np.random.default_rng(seed)
        for metric, weighted in (("weighted", True), ("standard", False)):
            t = time.time()
            L, C = vim1.ro_channels(base, R["init"], R["candB"], weighted=weighted)
            pred = vim1.fit_ro(D, L, C, rng, R["candB"])
            power = vim1.loo_power(D["val"], pred)
            for cname in ("candA", "candB"):
                choice, _ = vim1.identify_repeated(D["val"], pred, power, cand=R[cname])
                RO[lum, subj, metric, cname] = 100 * (choice == np.arange(120)).mean()
            print(f"{lum}: {subj} RO {metric}: O1a {RO[lum, subj, metric, 'candA']:.1f}%, O1b {RO[lum, subj, metric, 'candB']:.1f}% "
                  f"({time.time() - t:.0f} s)")
            del L, C
gwp_mean = {name: np.mean([100 * (ID[s, name]["choice"] == np.arange(120)).mean() for s in ("S1", "S2")]) for name in READINGS}
for lum in ("SM 8", "grey"):
    for cname in ("candA", "candB"):
        w_ = np.mean([RO[lum, s, "weighted", cname] for s in ("S1", "S2")])
        s_ = np.mean([RO[lum, s, "standard", cname] for s in ("S1", "S2")])
        print(f"{lum}, {cname}: mean over subjects, weighted {w_:.1f}% (paper 55, within 5.5: {abs(w_ - 55) <= 5.5}), "
              f"standard {s_:.1f}% (paper 42, within 4.2: {abs(s_ - 42) <= 4.2})")
print("GWP mean over subjects:", {k: round(v, 1) for k, v in gwp_mean.items()})

**Result: reproduced.** With the luminance channel as SM 8 defines it (absolute deviation from the
mean luminance), the RO model identifies, averaged over subjects, 52.1% of images with the
spatially-weighted metrics and 43.3% with the standard metrics (O1 a; 50.8% and 43.3% under O1 b),
within tolerance of 55% and 42%, and both below the GWP mean of 81.7% under O1a-O5a (and of 61.2%
under the O5 (b) readings). With the luminance relative to the grey background, the reading of the
first version, they are 53.3% and 43.8% (52.9% and 42.5% under O1 b), within tolerance as well.

## suppfig11-rf-size-eccentricity

> Receptive-field size increases with eccentricity, and from V1 to V2 to V3.

Criterion: RF size is the extent of +/-2 SD (4 SD) of the Gaussian fitted to the best initial
model's envelope; voxels with p < 0.01 whose RF lies wholly inside the stimulus; pooled across
subjects; 1 degree eccentricity bins (0.5 degree as a variant), median per bin with at least 10
voxels; reproduced when, per area, the Spearman correlation of size with eccentricity is positive
at p < 0.05, and in most bins shared by the three areas the medians order V1 < V2 < V3. "Inside the
stimulus" is read as inside the circular aperture, since Figure 2's legend calls the white circle
"the bounds of the stimulus". Four filters are shown: the +/-2 SD square inside the paper's 10 degree
radius and inside the mask's 9.5 degree radius (the reading adopted, SM 5 defining the RF as the
+/-2 SD square), a disc of 2 SD around the centre inside the mask's radius (the filter of the first
version of this notebook), and the +/-2 SD square inside the 20 degree square.

In [ ]:
rows = []
FILTERS = ("20 deg square", "disc in the mask circle", "square in the 10 deg circle", "square in the mask circle")
for subj in ("S1", "S2"):
    R = RES[subj]; D = R["D"]; init = R["init"]
    ecc = np.hypot(init["cx"], init["cy"]); size = 4 * init["sd"]
    corner = np.hypot(np.abs(init["cx"]) + 2 * init["sd"], np.abs(init["cy"]) + 2 * init["sd"])   # farthest corner of the +/-2 SD square
    filt = [(np.abs(init["cx"]) + 2 * init["sd"] <= 10) & (np.abs(init["cy"]) + 2 * init["sd"] <= 10),
            ecc + 2 * init["sd"] <= RADIUS, corner <= 10, corner <= RADIUS]
    for k in (1, 2, 3):
        s = (D["roi"] == k) & (init["p"] < 0.01)
        rows.append((subj, k, ecc[s], size[s], [f[s] for f in filt]))
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for j, (ax, bounds) in enumerate(zip(axes, FILTERS)):
    for width in (1.0, 0.5):
        bins = np.arange(0, 10 + width, width)
        med = {}
        for k in (1, 2, 3):
            e = np.concatenate([r[2][r[4][j]] for r in rows if r[1] == k]); z = np.concatenate([r[3][r[4][j]] for r in rows if r[1] == k])
            rho, p = stats.spearmanr(e, z)
            b = np.digitize(e, bins) - 1
            med[k] = {i: np.median(z[b == i]) for i in range(len(bins) - 1) if (b == i).sum() >= 10}
            if width == 1.0:
                print(f"{bounds} V{k}: {e.size} voxels, Spearman rho {rho:.3f}, p = {p:.2g}; size {z.min():.1f} to {z.max():.1f} deg, "
                      f"median {np.median(z):.2f}; by subject " + ", ".join(f"{s} {int((r[4][j]).sum())}" for s in ("S1", "S2") for r in rows if r[0] == s and r[1] == k))
                ax.plot(bins[list(med[k])] + width / 2, list(med[k].values()), "o-", label=f"V{k}")
        shared = sorted(set(med[1]) & set(med[2]) & set(med[3]))
        ordered = [med[1][i] < med[2][i] < med[3][i] for i in shared]
        print(f"   {bounds}, {width} deg bins: {sum(ordered)} of {len(shared)} shared bins ordered V1 < V2 < V3")
    ax.set_xlabel("eccentricity (deg)"); ax.set_ylabel("median RF size, 4 SD (deg)"); ax.set_title(f"RF: {bounds}"); ax.legend()
plt.tight_layout(); plt.show()

**Result: partially reproduced, resting on the reading of "inside the stimulus" as the circular
aperture.** Figure 2's legend calls the white circle "the bounds of the stimulus", so the circle is
the better supported reading. RF size increases with eccentricity in each area under every filter
(Spearman rho 0.391 to 0.676, p < 1e-9). With the +/-2 SD square inside the 10 degree circle, the
reading adopted, the medians order V1 < V2 < V3 in 2 of the 5 shared 1 degree bins and 1 of 7 shared
0.5 degree bins; inside the mask's 9.5 degree radius it is the same (2 of 5, 1 of 7), and with the
disc filter of the first version 3 of 6 and 3 of 9. Only with the 20 degree square and 1 degree bins
does the order hold in most bins (5 of 8; 6 of 12 with 0.5 degree bins). Under every filter the
median RF size over all eccentricities is smaller in V3 than in V1 and V2 (for the reading adopted,
3.44, 3.25 and 2.41 degrees). The first part of the claim holds and the area order does not.

## fig2-representative-voxel

> A representative V1 voxel of S1 (voxel 42205) has its receptive field in the left hemifield just
> below the horizontal meridian, broadband orientation tuning and high-pass spatial frequency tuning.

The data set's `voxIdx` holds 42205 for one S1 V1 voxel; it is taken to be the paper's voxel (the
paper does not say how it numbers voxels). Criterion: envelope peak left of the vertical meridian
and below the horizontal meridian within the central half of the stimulus height; SF tuning highest
at the highest frequency; orientation tuning maximum less than twice its minimum.

In [ ]:
GRAT = vim1.gratings()
W, table, M = W64
Xg = vim1.gwp_channels(GRAT.reshape(-1, 64, 64), W, table, np.ones(table.shape[0], bool), False)   # 640 x channels


def tuning_of(k, dc, keep):
    r = ((Xg * keep[None, :]) @ k + dc).reshape(GRAT.shape[:3]).mean(2)
    return r.mean(1), r.mean(0)


R = RES["S1"]; D = R["D"]
v = int(np.flatnonzero(D["vox_idx"] == 42205)[0])
init = R["init"]
print(f"voxel 42205: ROI {vim1.ROI_NAMES[D['roi'][v]]}, best initial model {init['best'][v]} deg, r = {init['r'][v]:.3f}, "
      f"p = {init['p'][v]:.3f}; Gaussian centre ({init['cx'][v]:.2f}, {init['cy'][v]:.2f}) deg, SD {init['sd'][v]:.2f} deg")
for reading in ("O5a", "O5b"):
    if reading == "O5a":
        if v not in R["kern_a"]:
            print("O5a: no final model (voxel not among the O1 a voxels)"); continue
        res_, k, dc, centre, fov, keep = R["kern_a"][v]
    else:
        name, k, dc = R["kern_b"][v]; keep = keeps_b[name]; fov = vim1.FINAL_B[name][1]; centre = (0.0, 0.0)
    env = vim1.final_envelope(k, M)
    iy, ix = np.unravel_index(np.argmax(env), env.shape)
    px_deg = fov / 64
    x_deg = centre[0] + (ix + 0.5 - 32) * px_deg; y_deg = centre[1] - (iy + 0.5 - 32) * px_deg
    ori, sf = tuning_of(k, dc, keep)
    sf_cpd = np.array([1, 2, 4, 8, 16]) / fov
    pr = np.corrcoef(D["val"][:, v], (R["pred_a"] if reading == "O5a" else R["pred_b"])[:, v])[0, 1]
    print(f"{reading}: FOV {fov:.2f} deg centred ({centre[0]:.2f}, {centre[1]:.2f}); predictive power r = {pr:.3f}")
    print(f"   envelope peak at ({x_deg:.2f}, {y_deg:.2f}) deg: left {x_deg < 0}, below {y_deg < 0}, within 5 deg of the horizontal meridian {abs(y_deg) <= 5}")
    gy, gx, gsd = vim1.fit_gaussian(env)
    gx_deg = centre[0] + (gx - 32) * px_deg; gy_deg = centre[1] - (gy - 32) * px_deg
    print(f"   Gaussian fitted to the envelope: centre ({gx_deg:.2f}, {gy_deg:.2f}) deg, SD {gsd * px_deg:.2f} deg: left {gx_deg < 0}, "
          f"below {gy_deg < 0}, within 5 deg of the horizontal meridian {abs(gy_deg) <= 5}")
    print("   orientation tuning (0 to 157.5 deg):", np.round(ori, 3), f"; max/min {ori.max() / ori.min():.2f}" if ori.min() > 0 else "; min <= 0",
          f"; without the DC offset {np.round(ori - dc, 3)}, max/min {(ori - dc).max() / (ori - dc).min():.2f}" if (ori - dc).min() > 0 else "; without DC min <= 0")
    print("   SF tuning at", np.round(sf_cpd, 2), "cycles/deg:", np.round(sf, 3), f"; highest at {sf_cpd[np.argmax(sf)]:.2f} cycles/deg, the highest frequency: {np.argmax(sf) == 4}")
    ap = vim1.aperture(fov, centre, RADIUS)
    Xap = vim1.gwp_channels((GRAT * ap).reshape(-1, 64, 64), W, table, keep, False)
    r_ap = (Xap @ k).reshape(GRAT.shape[:3]).mean(2)
    o_ap, s_ap = r_ap.mean(1), r_ap.mean(0)
    print(f"   variant, gratings inside the stimulus aperture ({ap.mean():.2f} of the FOV), without DC: orientation {np.round(o_ap, 3)}, "
          f"max/min {o_ap.max() / o_ap.min():.2f}; SF {np.round(s_ap, 3)}, highest at level {np.argmax(s_ap)}")
    print(f"   envelope near the peak (rows top to bottom, deg above the horizontal meridian of each row): "
          f"{[round(centre[1] - (i + 0.5 - 32) * px_deg, 2) for i in range(iy - 2, iy + 3)]}")
    print("   ", np.round(env[iy - 2:iy + 3, ix - 2:ix + 3] / env.max(), 3).tolist())

**Result: partially reproduced, resting on O5 (a) and tuning without the DC offset.** Under O5 (a),
whose 5.45 degree field of view gives a highest frequency of 2.94 cycles/deg as on the paper's axis
(2.9), the envelope peaks at (-3.42, +0.24) deg: in the left hemifield but 0.24 degrees above the
horizontal meridian (the initial model's Gaussian centre, -0.57, is below); SF tuning is highest at
the highest frequency (high-pass); the orientation tuning's max/min is 2.23, so not broadband by the
rule. Under O5 (b) the peak is at (-4.22, -0.47), SF tuning is highest at the highest frequency and
orientation max/min is 1.77: all three parts hold. A Gaussian fitted to the final envelope puts the
centre at (-3.60, +0.04) deg under O5 (a), also above the horizontal meridian, and at (-4.25, -0.48)
deg under O5 (b), below it, so the peak and the fitted centre agree on "below" under each reading. The ratios with the DC offset added (16.2 and
6.54) depend on the arbitrary grating amplitude and do not bear on the rule; the ratios without it
do not depend on the amplitude.

## suppfig7-roi-tuning

> The V1-averaged orientation tuning curve is nearly flat and the V1-averaged spatial frequency
> tuning curve is band-pass; voxels have only a slight orientation bias.

Criterion: mean over V1 voxels of the final models' grating tuning curves, bootstrap SE across
voxels; reproduced when, under one reading, the orientation curve's range is under 10% of its mean
and the SF curve peaks at an intermediate frequency above both its lowest and highest values by
more than 2 SE. Reading (a) of the models (final GWP); reading (b), model I of the constrained
models, is not run. Voxels: all V1 voxels with a final model, under O5 (a) and (b). The curves are
averaged over the five spatial frequency levels of each model (1 to 16 cycles per field of view),
since the fields of view differ between voxels.

In [ ]:
rng_t = np.random.default_rng(41)
Wd64 = W.toarray()
fig, axes = plt.subplots(2, 4, figsize=(14, 6))
col = 0
for subj in ("S1", "S2"):
    R = RES[subj]; D = R["D"]
    for reading in ("O5a", "O5b"):
        oris, sfs, dcs, oris_ap, sfs_ap = [], [], [], [], []
        for v, val in (R["kern_a"] if reading == "O5a" else R["kern_b"]).items():
            if D["roi"][v] != 1 or not R["candB"][v]:
                continue
            if reading == "O5a":
                _, k, dc, _, _, keep = val
            else:
                name, k, dc = val; keep = keeps_b[name]
            o, s = tuning_of(k, dc, keep)
            oris.append(o); sfs.append(s); dcs.append(dc)
            ap = vim1.aperture(val[4], val[3], RADIUS) if reading == "O5a" else vim1.aperture(vim1.FINAL_B[name][1], (0.0, 0.0), RADIUS)
            Xap = vim1.gwp_channels((GRAT * ap).reshape(-1, 64, 64), Wd64, table, keep, False)
            r_ap = (Xap @ k).reshape(GRAT.shape[:3]).mean(2)
            oris_ap.append(r_ap.mean(1)); sfs_ap.append(r_ap.mean(0))
        dcs = np.array(dcs)[:, None]
        for label, oo, ss in (("with DC", np.array(oris), np.array(sfs)), ("without DC", np.array(oris) - dcs, np.array(sfs) - dcs),
                              ("without DC, gratings inside the stimulus aperture", np.array(oris_ap), np.array(sfs_ap)),
                              ("median over voxels, without DC", np.array(oris) - dcs, np.array(sfs) - dcs)):
            if label.startswith("median"):
                mo, ms = np.median(oo, 0), np.median(ss, 0)
                print(f"{subj} {reading} {label}: orientation {np.round(mo, 3)}, range/median {100 * (mo.max() - mo.min()) / abs(mo.mean()):.1f}%; "
                      f"SF {np.round(ms, 3)}, peak at level {int(np.argmax(ms))}")
                continue
            mo, ms = oo.mean(0), ss.mean(0)
            boot = rng_t.integers(0, len(oo), (1000, len(oo)))
            se_o = oo[boot].mean(1).std(0); se_s = ss[boot].mean(1).std(0)
            rng_ratio = (mo.max() - mo.min()) / abs(mo.mean())
            peak = int(np.argmax(ms))
            bandpass = 0 < peak < 4 and ms[peak] - ms[0] > 2 * se_s[peak] and ms[peak] - ms[4] > 2 * se_s[peak]
            print(f"{subj} {reading} {label}: {len(oo)} V1 voxels; orientation curve {np.round(mo, 3)} (SE up to {se_o.max():.3f}), "
                  f"range/mean {100 * rng_ratio:.1f}% (flat by the rule: {rng_ratio < 0.1})")
            print(f"   SF curve (1 to 16 cycles/FOV) {np.round(ms, 3)} (SE {np.round(se_s, 3)}); peak at level {peak}; band-pass by the rule: {bandpass}")
            if label == "without DC":
                axes[0, col].errorbar(np.arange(8) * 22.5, mo, se_o); axes[0, col].set_title(f"{subj} {reading} orientation")
                axes[1, col].errorbar([1, 2, 4, 8, 16], ms, se_s); axes[1, col].set_xscale("log"); axes[1, col].set_title(f"{subj} {reading} SF")
                axes[1, col].set_xlabel("cycles per FOV")
        if reading == "O5a":
            fovs = np.array([val[4] for v, val in R["kern_a"].items() if D["roi"][v] == 1])
            print(f"   fields of view {fovs.min():.1f} to {fovs.max():.1f} deg (median {np.median(fovs):.1f}): level 16 cycles/FOV is "
                  f"{16 / np.median(fovs):.2f} cycles/deg at the median")
        col += 1
plt.tight_layout(); plt.show()

**Result: not reproduced, for the readings run.** No reading run gives a band-pass V1 spatial
frequency curve by the rule, and in every reading the orientation curve's range is far above 10% of
its mean (31% at the least). Under O5 (b), and under O5 (a) with gratings inside the aperture or
with the median over voxels, the curve rises to the highest level (16 cycles per field of view),
so it is high-pass. Under O5 (a) with full-field gratings the mean curves are dominated by a few
voxels with very large responses (SE up to about 210,000 in the units of the responses); there the
S1 curve rises to the highest level and the S2 curve peaks at level 2 but is not above its ends by
2 SE. Gratings inside the aperture remove those voxels; the curves change shape (S1:
[0.145, 0.257, 0.483, 0.801, 0.935]) and still rise to the highest level. The curves with the DC
offset depend on the arbitrary grating amplitude and do not bear on the rule. Reading (b) of the
models (model I of the constrained models) and reading (b) of the voxels were not run, so the
outcome holds only for the readings run.

## suppfig4-500-voxels

> Identification performance rises with the number of voxels used and is best at about 500 voxels
> for both subjects, with repeated and single trials.

Criterion: P5 at set size 120 with 1 to 1,000 voxels; in each of the four conditions (S1 and S2,
repeated and single trials), performance at 500 voxels is the highest of the nine or within 3
points of it, and above that at 50; partially reproduced when this holds in two or three
conditions. The single-trial conditions are blocked (fig4a-single-trial); the repeated-trial
conditions are below.

In [ ]:
NV = [1, 5, 10, 20, 50, 100, 200, 500, 1000]
for subj in ("S1", "S2"):
    D = RES[subj]["D"]
    for name in READINGS:
        I = ID[subj, name]
        perf = [100 * (vim1.identify_repeated(D["val"], I["pred"], I["power"], n_vox=n, cand=I["cand"])[0] == np.arange(120)).mean() for n in NV]
        print(f"{subj} {name} ({I['cand'].sum()} candidates):", dict(zip(NV, np.round(perf, 1))))

**Result: partially reproduced.** In both repeated-trial conditions (O1a-O5a) the rule holds: S1
reaches 90.8% at 500 voxels, the highest of the nine (also 90.8% at 1,000), against 61.7% at 50; S2
reaches 72.5% at 500, within 3 points of the highest (74.2% at 200), against 45.8% at 50. It holds
under the other three readings as well (500 voxels the highest or within 1.7 points of it, and above
50). The two single-trial conditions are blocked, so the rule holds in two of the four conditions.


## Summary

| Claim | Outcome |
|---|---|
| fig3-s1-repeated-120 | reproduced (109/120 under O1a-O5a) |
| fig4a-s2-repeated-120 | reproduced (87/120 under O1a-O5a) |
| fig4a-single-trial | blocked, too large |
| fig4a-noise-ceiling | blocked, too large (depends on single trials) |
| fig4b-set-size-1000 | blocked, data missing (999-image library) |
| fig4b-extrapolation | blocked, data missing |
| fig4c-retinotopy-only-extrapolation | blocked, data missing |
| suppm8-retinotopy-only-120 | reproduced |
| suppfig8-constrained-models | blocked, data missing |
| time-two-months | blocked, data missing |
| time-fourteen-months | blocked, data missing |
| errors-similar-images | reproduced (local contrast reading) |
| suppfig4-500-voxels | partially reproduced (both repeated-trial conditions; single trials blocked) |
| supptable1-voxel-counts | reproduced |
| supptable1-predictive-power-by-area | reproduced (assumed) |
| supptable1-snr | blocked, too large |
| methods-v4-poor | reproduced |
| fig2-representative-voxel | partially reproduced (assumed) |
| suppfig7-roi-tuning | not reproduced |
| suppfig10-retinotopy-validation | blocked, data missing |
| suppfig11-rf-size-eccentricity | partially reproduced (assumed) |
| suppdisc1-classification-extension | not reproduced |
| suppm7-selected-voxels-v1 | not reproduced |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning
and the differences from the paper's method.

How the analysis differs from the paper's: one gradient-descent fit per model instead of the mean of
100 bootstrap fits; one stopping set shared by the voxels fitted together; the initial fields of
view are 10.0 and 5.12 degrees (the paper's 10.1 and 5.2 are not on its resolution list); the
wavelet SD follows from a 1-octave bandwidth (orientation bandwidth 37 degrees, not 41); stimulus
values taken as linear in luminance; area-averaging resampling; the final model's field of view
read two ways (O5), of which only the +/-2 SD square reproduces the identification results. The single-trial and SNR analyses need per-voxel fits of
the time series, which were not run in this notebook. The retinotopy-only model's luminance channel
is taken as SM 8 defines it, relative to the mean luminance; relative to the grey background, as
in the first version, the outcome is the same.

What was found about the data set: the repeated-trial responses, ROI labels and stimuli match the
paper (voxel counts per area equal Supplementary Table 1; run designs as SM 1 describes). The readme
calls `Stimuli.mat` a MATLAB 7.3 file (its header says MATLAB 5.0) and gives `Stimuli_Trn_FullRes_15.mat` as
70 x 120 x 120 (it is 70 x 500 x 500). 345 (S1) and 214 (S2) V1 to V3 voxels have NaN responses,
which the readme does not mention. The NIfTI headers give a TR of 0. The data set holds no
single-trial responses, no standard errors, none of S1's later sessions, no 999-image library and
no retinotopic mapping data.

In [ ]:
print(f"total run time {(time.time() - STARTED) / 60:.1f} minutes; peak memory {peak_gb():.1f} GB")